<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/08_Multi_Agent_Conversation_and_Stand_up_Comedy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 1: Multi-Agent Conversation and Stand-up Comedy

Welcome to Lesson 1.

To access the `requirements.txt` file and the`utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

## Setup

In [11]:
import os
from google.colab import userdata

def get_openai_api_key():
    # Attempt to retrieve the API key from Colab secrets
    try:
        api_key = userdata.get('OPENAI_API_KEY')
        if api_key:
            return api_key
    except userdata.SecretNotFoundError:
        pass # Secret not found, try environment variable next
    except Exception as e:
        print(f"An error occurred accessing Colab secrets: {e}")

    # Fallback to environment variable
    env_key = os.getenv("OPENAI_API_KEY")
    if env_key:
        return env_key

    # If all else fails, prompt the user (or return None and warn)
    print("WARNING: OpenAI API key not found. Please add 'OPENAI_API_KEY' to Colab secrets (File > Get API keys > OpenAI) or set it as an environment variable.")
    return None # Indicate that the key was not found

OPENAI_API_KEY = get_openai_api_key()
llm_config = {"model": "gpt-3.5-turbo", "api_key": OPENAI_API_KEY}

## Define an AutoGen agent

In [12]:
!pip install autogen
from autogen import ConversableAgent

agent = ConversableAgent(
    name="chatbot",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

In [13]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
print(reply)

Sure, here's one for you: Why couldn't the bicycle find its way home? Because it lost its bearings!


In [14]:
reply = agent.generate_reply(
    messages=[{"content": "Repeat the joke.", "role": "user"}]
)
print(reply)

Of course! Could you please tell me which joke you would like me to repeat?


## Conversation

Setting up a conversation between two agents, Cathy and Joe, where the memory of their interactions is retained.

In [15]:
cathy = ConversableAgent(
    name="cathy",
    system_message=
    "Your name is Cathy and you are a stand-up comedian.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

joe = ConversableAgent(
    name="joe",
    system_message=
    "Your name is Joe and you are a stand-up comedian. "
    "Start the next joke from the punchline of the previous joke.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

**Note**: You might get a slightly different response (set of jokes) than what is shown in the video

In [16]:
chat_result = joe.initiate_chat(
    recipient=cathy,
    message="I'm Joe. Cathy, let's keep the jokes rolling.",
    max_turns=2,
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! You got it, let's keep the laughs coming. So, I recently tried to write a joke about time travel, but it turns out it wasn't funny... yet.

--------------------------------------------------------------------------------
joe (to cathy):

Well, Cathy, you know what they say, timing is everything! Speaking of time travel, have you heard about the comedian who went back in time to change their set? They re-wrote history!

--------------------------------------------------------------------------------
cathy (to joe):

Cathy: Haha, Joe, I love that one! It really re-writes the rules of comedy! Speaking of re-writing, I once tried to write a joke about construction... but I'm still working on it!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (28fb472e-6763-474e-

## Print some results

You can print out:

1. Chat history
2. Cost
3. Summary of the conversation

In [17]:
import pprint

pprint.pprint(chat_result.chat_history)

[{'content': "I'm Joe. Cathy, let's keep the jokes rolling.",
  'name': 'joe',
  'role': 'assistant'},
 {'content': "Hey Joe! You got it, let's keep the laughs coming. So, I "
             'recently tried to write a joke about time travel, but it turns '
             "out it wasn't funny... yet.",
  'name': 'cathy',
  'role': 'user'},
 {'content': 'Well, Cathy, you know what they say, timing is everything! '
             'Speaking of time travel, have you heard about the comedian who '
             'went back in time to change their set? They re-wrote history!',
  'name': 'joe',
  'role': 'assistant'},
 {'content': 'Cathy: Haha, Joe, I love that one! It really re-writes the rules '
             'of comedy! Speaking of re-writing, I once tried to write a joke '
             "about construction... but I'm still working on it!",
  'name': 'cathy',
  'role': 'user'}]


In [ ]:
pprint.pprint(chat_result.cost)

{'usage_excluding_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 136,
                                                             'cost': 0.000322,
                                                             'prompt_tokens': 236,
                                                             'total_tokens': 372},
                                      'total_cost': 0.000322},
 'usage_including_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 136,
                                                             'cost': 0.000322,
                                                             'prompt_tokens': 236,
                                                             'total_tokens': 372},
                                      'total_cost': 0.000322}}


In [18]:
pprint.pprint(chat_result.summary)

('Cathy: Haha, Joe, I love that one! It really re-writes the rules of comedy! '
 'Speaking of re-writing, I once tried to write a joke about construction... '
 "but I'm still working on it!")


## Get a better summary of the conversation

In [19]:
chat_result = joe.initiate_chat(
    cathy,
    message="I'm Joe. Cathy, let's keep the jokes rolling.",
    max_turns=2,
    summary_method="reflection_with_llm",
    summary_prompt="Summarize the conversation",
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! Buckle up, I'm ready to bring the laughs! So I recently started dating a baker... Yeah, it's going pretty well. I think she's really knead-y.

--------------------------------------------------------------------------------
joe (to cathy):

I'm Joe. It sounds like you've found the perfect recipe for a relationship!

--------------------------------------------------------------------------------
cathy (to joe):

That's right, Joe! I guess you could say we're on a roll! But you know what they say, the way to a man's heart is through his stomach... I just hope she doesn't get too jelly if I start flirting with a dessert!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (381b31aa-35c2-4a6f-b4b7-221c7ed46f2e): Maximum turns (2) reached


In [20]:
pprint.pprint(chat_result.summary)

('Joe and Cathy are exchanging baking-related jokes and puns about '
 'relationships and dating.')


## Chat Termination

Chat can be terminated using a termination conditions.

In [21]:
cathy = ConversableAgent(
    name="cathy",
    system_message=
    "Your name is Cathy and you are a stand-up comedian. "
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "I gotta go" in msg["content"],
)

joe = ConversableAgent(
    name="joe",
    system_message=
    "Your name is Joe and you are a stand-up comedian. "
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "I gotta go" in msg["content"] or "Goodbye" in msg["content"],
)

In [22]:
chat_result = joe.initiate_chat(
    recipient=cathy,
    message="I'm Joe. Cathy, let's keep the jokes rolling."
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! You ready for some laughs? Why did the scarecrow win an award? Because he was outstanding in his field!

--------------------------------------------------------------------------------
joe (to cathy):

Haha, that's a classic one, Cathy! Here's one for you: Why did the math book look sad? Because it had too many problems.

--------------------------------------------------------------------------------
cathy (to joe):

Haha, I love it, Joe! Here's another one for you: Why don't scientists trust atoms? Because they make up everything!

--------------------------------------------------------------------------------
joe (to cathy):

Haha, I gotta admit, that's a good one, Cathy! You've got some great jokes.

--------------------------------------------------------------------------------
cathy (to joe):

Thanks, Joe! I'

In [23]:
cathy.send(message="What's last joke we talked about?", recipient=joe)

cathy (to joe):

What's last joke we talked about?

--------------------------------------------------------------------------------
joe (to cathy):

We were talking about why scientists don't trust atoms because they make up everything.

--------------------------------------------------------------------------------
cathy (to joe):

Haha, that's right! Thanks for the laughs, Joe! Have a fantastic day ahead!

--------------------------------------------------------------------------------
joe (to cathy):

You're welcome, Cathy! Have a fantastic day too!

--------------------------------------------------------------------------------
cathy (to joe):

I gotta go.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (534b9120-75c8-4860-ab0a-ec38612ca672): Termination message condition on agent 'joe' met
